<a href="https://colab.research.google.com/github/007arjungangwar/Foundations-of-Data-Engineering-Sec1/blob/main/Create_complete_pipeline_flow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

250 movie records fetched from RapidAPI.

In [ ]:

# 1. IMPORT LIBRARIES
import requests
import pandas as pd
import json
import time

from google.colab import userdata
from sqlalchemy import text

# 2. EXTRACT DATA FROM RAPIDAPI

def extract_data():

    print("Extracting data from RapidAPI...")

    Rapidapi_key = userdata.get("Rapidapi-key")

    url = "https://imdb236.p.rapidapi.com/api/imdb/top250-movies"

    headers = {
        "x-rapidapi-key": Rapidapi_key,
        "x-rapidapi-host": "imdb236.p.rapidapi.com"
    }

    response = requests.get(
        url,
        headers=headers,
        timeout=30
    )

    response.raise_for_status()

    data = response.json()

    print("Extraction completed!")
    print("Records extracted:", len(data))

    return data


# 3. TRANSFORM DATA
def transform_data(data):

    print("Transforming data...")

    df = pd.DataFrame(data)

    # Convert dictionaries and lists into JSON strings
    for col in df.columns:

        df[col] = df[col].apply(
            lambda x: json.dumps(x)
            if isinstance(x, (dict, list))
            else x
        )

    # Handle missing values
    df = df.where(pd.notnull(df), None)

    print("Transformation completed!")
    print("DataFrame shape:", df.shape)

    return df


# 4. LOAD DATA INTO SUPABASE
def load_data(df):

    print("Loading data into Supabase...")

    df.to_sql(
        name="imdb_top250",
        con=engine,
        schema="public",
        if_exists="replace",
        index=False
    )

    print("Data loaded successfully!")


# 5. VERIFY DATA
def verify_data():

    print("Verifying database...")

    query = """
    SELECT COUNT(*) AS total_records
    FROM imdb_top250;
    """

    df_check = pd.read_sql(query, engine)

    print(df_check)

    return df_check


# 6. COMPLETE ETL PIPELINE
def run_pipeline():

    start_time = time.time()

    print("=" * 50)
    print("IMDb TOP 250 ETL PIPELINE STARTED")
    print("=" * 50)

    # Extract
    data = extract_data()

    # Transform
    df = transform_data(data)

    # Load
    load_data(df)

    # Verify
    verify_data()

    execution_time = time.time() - start_time

    print("=" * 50)
    print("PIPELINE COMPLETED SUCCESSFULLY!")
    print("Execution time:", round(execution_time, 2), "seconds")
    print("=" * 50)

    return df


# 7. RUN PIPELINE
df = run_pipeline()

# **OYO Chennai Hotels ETL Pipeline**

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

from sqlalchemy import text

def extract_data():

    print("Starting web scraping...")

    hotel_data = []

    for page_num in range(1, 6):

        url = f"https://www.oyorooms.com/hotels-in-chennai/?page={page_num}"

        response = requests.get(
            url,
            headers={"User-Agent": "Mozilla/5.0"},
            timeout=30
        )

        print(f"Scraping page {page_num}")

        if response.status_code != 200:
            print("Request failed:", response.status_code)
            break

        soup = BeautifulSoup(response.content, "html.parser")

        hotels = soup.find_all(
            "div",
            class_="oyo-cell--12-col oyo-cell--8-col-tablet oyo-cell--4-col-phone"
        )

        for hotel in hotels:

            title = hotel.find(
                "h3",
                class_="listingHotelDescription__hotelName d-textEllipsis"
            )

            address = hotel.find(
                "span",
                class_="u-line--clamp-2"
            )

            try:
                star_rating = hotel.find_all("span")[2].find("span").text.strip()
            except:
                star_rating = None

            try:
                number_rating = hotel.find(
                    "span",
                    class_="hotelRating__ratingSummary hotelRating__rating--clickable"
                ).text.strip("()")
            except:
                number_rating = None

            try:
                price = hotel.find(
                    "span",
                    class_="listingPrice__finalPrice listingPrice__finalPrice--black"
                ).text.strip().replace("₹", "").replace(",", "")
            except:
                price = None

            hotel_data.append({
                "Title": title.text.strip() if title else None,
                "Address": address.text.strip() if address else None,
                "Star_rating": star_rating,
                "Number_rating": number_rating,
                "Price": price
            })

        time.sleep(2)

    print("Extraction completed!")
    print("Hotels extracted:", len(hotel_data))

    return hotel_data


def transform_data(hotel_data):
    print("Transforming data...")

    df = pd.DataFrame(hotel_data)

    # Convert Price into numeric
    df["Price"] = pd.to_numeric(
        df["Price"],
        errors="coerce"
    )

    # Remove duplicate hotels
    df = df.drop_duplicates(subset=["Title", "Address"])

    # Reset index
    df = df.reset_index(drop=True)

    print("Transformation completed!")
    print("Data shape:", df.shape)

    return df


# def save_csv(df):

#     df.to_csv(
#         "chennai_oyo_hotels.csv",
#         index=False
#     )

#     print("CSV file saved successfully!")


def load_data(df):
    print("Loading data into Supabase...")

    df.to_sql(
        name="hotel_data_t2",
        con=engine,
        schema="public",
        if_exists="replace", # append
        index=False
    )

    print("Data loaded successfully!")

def verify_data():

    query = """
    SELECT COUNT(*) AS total_hotels
    FROM hotel_data_t2;
    """

    df_check = pd.read_sql(query, engine)

    print("Database verification:")
    print(df_check)

    return df_check


# Run the complete pipeline
def run_pipeline():

    start_time = time.time()

    print("=" * 40)
    print("OYO HOTEL ETL PIPELINE STARTED")
    print("=" * 40)

    # Extract
    hotel_data = extract_data()

    # Transform
    df = transform_data(hotel_data)

    # Save CSV
    save_csv(df)

    # Load
    load_data(df)

    # Verify
    verify_data()

    execution_time = time.time() - start_time

    print("=" * 40)
    print("PIPELINE COMPLETED SUCCESSFULLY!")
    print("Execution time:", round(execution_time, 2), "seconds")
    print("=" * 40)

    return df

# **NewsAPI ETL Pipeline**

In [ ]:
# Step 1 — Import libraries
import requests
import pandas as pd
import time

from google.colab import userdata
from sqlalchemy import text


# Step 2 — Extract data from NewsAPI
def extract_data():

    print("Extracting news data...")

    Newsapi_key = userdata.get("Newsapi_key")

    url = "https://newsapi.org/v2/top-headlines"

    params = {
        "country": "us",
        "apiKey": Newsapi_key
    }

    response = requests.get(
        url,
        params=params,
        timeout=30
    )

    response.raise_for_status()

    data = response.json()

    articles = data["articles"]

    print("Extraction completed!")
    print("Total articles:", len(articles))

    return articles


# Step 3 — Transform data
def transform_data(articles):

    print("Transforming data...")

    df = pd.DataFrame(articles)

    # Extract source name

    df["source"] = df["source"].apply(lambda x: x["name"] if x else None)
    # df["source"] = df["source"].apply(
    #     lambda x: x["name"] if isinstance(x, dict) else None
    # )

    # Convert publication date
    df["publishedAt"] = pd.to_datetime(
        df["publishedAt"],
        errors="coerce"
    )

    # Remove duplicate articles
    df = df.drop_duplicates(subset=["url"])

    # Reset index
    df = df.reset_index(drop=True)

    print("Transformation completed!")
    print("Data shape:", df.shape)

    return df


def save_csv(df):
    df.to_csv(
        "news_data.csv",
        index=False
    )

    print("CSV file saved successfully!")

# Step 5 — Load into Supabase PostgreSQL
def load_data(df):

    print("Loading data into Supabase...")

    df.to_sql(
        name="news_data",
        con=engine,
        schema="public",
        if_exists="replace", # append
        index=False
    )

    print("Data loaded successfully!")


# Step 6 — Verify data
def verify_data():

    print("Verifying data in PostgreSQL...")

    query = """
    SELECT COUNT(*) AS total_articles
    FROM news_data;
    """

    df_check = pd.read_sql(query, engine)

    print(df_check)

    return df_check


# Step 7 — Complete ETL pipeline
def run_pipeline():

    start_time = time.time()

    # Extract
    articles = extract_data()

    # Transform
    df = transform_data(articles)

    # # Save CSV
    # save_csv(df)

    # Load into Supabase
    load_data(df)

    # Verify
    verify_data()

    execution_time = time.time() - start_time

    print("=" * 50)
    print("PIPELINE COMPLETED SUCCESSFULLY!")
    print("Execution time:", round(execution_time, 2), "seconds")
    print("=" * 50)

    return df